# GLiNER2.5 Multi — Uzbek exact-span NER

Экспериментальный notebook для fine-tuning **`fastino/gliner2.5-multi-v1`** на текущем Uzbek NER датасете.

Основные отличия от `02_train_uzbek_ner.ipynb`:

- GLiNER2.5 boundary architecture вместо BIO token-classification;
- единый `ExperimentConfig` для сетки экспериментов;
- управляемые доли morphology / case / script-swap / hard-negative synthetic data;
- чистый `dev` никогда не смешивается с synthetic;
- строгая проверка JSONL, offsets, overlap и dataset manifest SHA-256;
- punctuation-aware splitter, одинаковый при fine-tuning и inference;
- безопасная конвертация exact `(start, end)` разметки в GLiNER entity-training format;
- early stopping, один eval на эпоху, ограниченное число checkpoint'ов;
- native confidence для каждой сущности + подбор **per-label thresholds** на dev;
- exact-span micro/macro F1 и сверка с `scripts/evaluate.py`;
- error analysis: `wrong_label / boundary / spurious / missed`;
- все артефакты эксперимента сохраняются в `artifacts/experiments/<experiment_name>/`.

> **Важно:** GLiNER2 training format задаёт сущности строками, а не offsets. Поэтому ниже есть специальная защита от неоднозначных повторных mention'ов: если одна и та же строка встречается несколько раз, chunking делается так, чтобы GLiNER не создал ложную supervision-разметку.

## 0. Установка зависимостей

GLiNER2.5 требует Python 3.10+. Версия зафиксирована, чтобы API notebook не «поплыл» между экспериментами.

In [ ]:
import sys

assert sys.version_info >= (3, 10), f"Нужен Python >= 3.10, сейчас: {sys.version}"
print(sys.version)

In [ ]:
# Запустить один раз в новом окружении, затем перезапустить kernel.
# %pip install -q "gliner2[local,train]==2.0.0" pandas tqdm matplotlib

## 1. Единый конфиг эксперимента

Для clean baseline поставьте все `*_ratio = 0.0`.

`morphology_ratio=0.25` означает: детерминированно взять synthetic документов в количестве `25%` от числа **real train** документов. Это сознательно осторожнее старого notebook, где morphology почти удваивала train.

In [ ]:
from __future__ import annotations

import dataclasses
import hashlib
import inspect
import itertools
import json
import math
import os
import random
import re
import subprocess
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable, Iterator, Sequence

import pandas as pd

LABELS = ("ORG", "NAME", "GEO")
DATA_TO_MODEL = {
    "ORG": "organization",
    "NAME": "person",
    "GEO": "location",
}
MODEL_TO_DATA = {v: k for k, v in DATA_TO_MODEL.items()}
MODEL_LABELS = tuple(DATA_TO_MODEL[x] for x in LABELS)

ENTITY_DESCRIPTIONS = {
    "organization": "Named organizations, companies, institutions, government bodies, teams, or other named groups.",
    "person": "Names of people or named individuals.",
    "location": "Countries, regions, cities, settlements, geographic areas, or other named places.",
}

@dataclass
class ExperimentConfig:
    # --- model / reproducibility ---
    model_name: str = "fastino/gliner2.5-multi-v1"
    require_boundary_architecture: bool = True
    experiment_name: str | None = None
    seed: int = 42
    device: str = "auto"          # auto | cuda | mps | cpu
    precision: str = "auto"       # auto | fp32 | fp16 | bf16

    # --- data / schema ---
    word_splitter: str = "punctuation_aware"  # punctuation_aware | whitespace
    use_entity_descriptions: bool = True
    verify_manifest_sha256: bool = True

    # GLiNER uses word-level training records; this is NOT encoder subword length.
    train_chunk_words: int = 256
    max_gold_per_label: int = 28   # below current boundary capacity (32)
    unrepresentable_policy: str = "drop_record"  # drop_record | raise

    # Synthetic ratios relative to number of real train documents.
    morphology_ratio: float = 0.25
    case_ratio: float = 0.0
    script_swap_ratio: float = 0.0
    hard_negative_ratio: float = 0.0

    # --- training ---
    num_epochs: int = 8
    train_batch_size: int = 1
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 16
    encoder_lr: float = 1e-5
    task_lr: float = 5e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    scheduler_type: str = "cosine"
    warmup_ratio: float = 0.10
    gradient_checkpointing: bool = True

    eval_strategy: str = "epoch"
    early_stopping: bool = True
    early_stopping_patience: int = 2
    save_total_limit: int = 1

    # Optional PEFT. Full FT is default for the main comparison.
    use_lora: bool = False
    lora_r: int = 16
    lora_alpha: float = 32.0
    lora_dropout: float = 0.0
    lora_target_modules: tuple[str, ...] = ("encoder",)
    # Keep False so best/final are directly reloadable full checkpoints.
    save_adapter_only: bool = False

    # --- inference / threshold tuning ---
    inference_chunk_words: int = 256
    inference_chunk_overlap: int = 64
    proposal_threshold: float = 0.05
    default_threshold: float = 0.50
    threshold_start: float = 0.10
    threshold_end: float = 0.90
    threshold_step: float = 0.05
    threshold_search_mode: str = "coordinate"  # coordinate | exhaustive
    threshold_coordinate_rounds: int = 2

    # --- debug / runtime ---
    preflight_examples: int = 512
    debug_train_records: int | None = None
    debug_dev_records: int | None = None
    run_official_scorer: bool = True

CFG = ExperimentConfig()


def validate_config(cfg: ExperimentConfig) -> None:
    ratio_fields = [
        "morphology_ratio", "case_ratio", "script_swap_ratio", "hard_negative_ratio"
    ]
    for name in ratio_fields:
        value = getattr(cfg, name)
        if value < 0:
            raise ValueError(f"{name} должен быть >= 0, получено {value}")

    if cfg.train_chunk_words <= 0 or cfg.inference_chunk_words <= 0:
        raise ValueError("chunk_words должен быть > 0")
    if not 0 <= cfg.inference_chunk_overlap < cfg.inference_chunk_words:
        raise ValueError("inference_chunk_overlap должен быть в [0, inference_chunk_words)")
    if not 0 <= cfg.proposal_threshold <= 1:
        raise ValueError("proposal_threshold должен быть в [0, 1]")
    if not 0 <= cfg.default_threshold <= 1:
        raise ValueError("default_threshold должен быть в [0, 1]")
    if cfg.threshold_step <= 0:
        raise ValueError("threshold_step должен быть > 0")
    if cfg.threshold_start > cfg.threshold_end:
        raise ValueError("threshold_start > threshold_end")
    if cfg.proposal_threshold > cfg.threshold_start + 1e-12:
        raise ValueError(
            "proposal_threshold должен быть <= минимального threshold в grid, "
            "иначе кандидаты будут потеряны до threshold search"
        )
    if cfg.threshold_search_mode not in {"coordinate", "exhaustive"}:
        raise ValueError("threshold_search_mode: coordinate | exhaustive")
    if cfg.unrepresentable_policy not in {"drop_record", "raise"}:
        raise ValueError("unrepresentable_policy: drop_record | raise")

validate_config(CFG)
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

## 2. Пути, seed и директория артефактов

Notebook сам ищет корень проекта — его можно запускать как из `notebooks/`, так и из корня репозитория.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "train.jsonl").exists() and (p / "data" / "dev.jsonl").exists():
            return p
    raise FileNotFoundError(
        "Не найден корень проекта с data/train.jsonl и data/dev.jsonl. "
        f"Текущая директория: {start}"
    )

PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "train.jsonl"
DEV_PATH = DATA_DIR / "dev.jsonl"
MANIFEST_PATH = DATA_DIR / "dataset_manifest.json"
AUG_DIR = DATA_DIR / "augmentation"


def ratio_tag(x: float) -> str:
    return f"{x:.3f}".rstrip("0").rstrip(".").replace(".", "p")


def auto_experiment_name(cfg: ExperimentConfig) -> str:
    model = cfg.model_name.split("/")[-1].replace(".", "_")
    return (
        f"{model}"
        f"__m{ratio_tag(cfg.morphology_ratio)}"
        f"_c{ratio_tag(cfg.case_ratio)}"
        f"_s{ratio_tag(cfg.script_swap_ratio)}"
        f"_h{ratio_tag(cfg.hard_negative_ratio)}"
        f"__seed{cfg.seed}"
    )

EXPERIMENT_NAME = CFG.experiment_name or auto_experiment_name(CFG)
ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "experiments" / EXPERIMENT_NAME
TRAINING_DIR = ARTIFACT_DIR / "training"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
os.environ["PYTHONHASHSEED"] = str(CFG.seed)

(ARTIFACT_DIR / "config.json").write_text(
    json.dumps(asdict(CFG), indent=2, ensure_ascii=False), encoding="utf-8"
)

print("PROJECT_DIR    =", PROJECT_DIR)
print("EXPERIMENT     =", EXPERIMENT_NAME)
print("ARTIFACT_DIR   =", ARTIFACT_DIR)

## 3. Строгая загрузка и проверка исходной разметки

Проверяем:

- уникальный `hash`;
- допустимые labels;
- character offsets;
- отсутствие duplicate/overlap gold spans;
- SHA-256 исходных `train/dev`, если есть `dataset_manifest.json`.

Synthetic применяется **только к train**.

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                raise ValueError(f"{path}:{line_no}: пустая строка")
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"{path}:{line_no}: invalid JSON: {e}") from e
    return records


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def validate_records(
    records: Sequence[dict[str, Any]],
    name: str,
    *,
    require_unique_hash: bool = True,
) -> dict[str, Any]:
    seen_hashes = set()
    counts = Counter()
    empty_records = 0

    for i, record in enumerate(records):
        h = record.get("hash")
        text = record.get("text")
        entities = record.get("entities")

        if not isinstance(h, str) or not h:
            raise ValueError(f"{name}[{i}]: invalid hash")
        if require_unique_hash and h in seen_hashes:
            raise ValueError(f"{name}[{i}]: duplicate hash {h}")
        seen_hashes.add(h)

        if not isinstance(text, str):
            raise ValueError(f"{name}[{i}]: text должен быть str")
        if not isinstance(entities, list):
            raise ValueError(f"{name}[{i}]: entities должен быть list")

        exact_seen = set()
        positional = []
        for j, ent in enumerate(entities):
            label = ent.get("label")
            start = ent.get("start")
            end = ent.get("end")

            if label not in LABELS:
                raise ValueError(f"{name}[{i}].entities[{j}]: bad label={label!r}")
            if type(start) is not int or type(end) is not int:
                raise ValueError(f"{name}[{i}].entities[{j}]: offsets должны быть int")
            if not 0 <= start < end <= len(text):
                raise ValueError(
                    f"{name}[{i}].entities[{j}]: bad span {start}:{end}, len={len(text)}"
                )

            key = (label, start, end)
            if key in exact_seen:
                raise ValueError(f"{name}[{i}]: duplicate entity {key}")
            exact_seen.add(key)
            positional.append((start, end, label))
            counts[label] += 1

        positional.sort()
        for (s1, e1, _), (s2, e2, _) in zip(positional, positional[1:]):
            if s2 < e1:
                raise ValueError(f"{name}[{i}]: overlapping gold entities")

        if not entities:
            empty_records += 1

    return {
        "records": len(records),
        "entities": sum(counts.values()),
        "empty_records": empty_records,
        "by_label": dict(counts),
    }


def verify_manifest(manifest_path: Path, train_path: Path, dev_path: Path) -> dict[str, Any] | None:
    if not manifest_path.exists():
        warnings.warn(f"Manifest не найден: {manifest_path}; SHA-проверка пропущена")
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    # Поддерживаем несколько простых shapes manifest без жёсткой привязки к одному формату.
    text = json.dumps(manifest)
    actual = {
        "train": sha256_file(train_path),
        "dev": sha256_file(dev_path),
    }
    missing = []
    for split, digest in actual.items():
        if digest not in text:
            missing.append((split, digest))
    if missing:
        raise RuntimeError(
            "SHA-256 исходных данных не совпадает с dataset_manifest.json: "
            + ", ".join(f"{s}={d}" for s, d in missing)
        )
    return actual

raw_train = read_jsonl(TRAIN_PATH)
raw_dev = read_jsonl(DEV_PATH)

train_stats = validate_records(raw_train, "train")
dev_stats = validate_records(raw_dev, "dev")

if CFG.verify_manifest_sha256:
    manifest_sha = verify_manifest(MANIFEST_PATH, TRAIN_PATH, DEV_PATH)
else:
    manifest_sha = None

if CFG.debug_train_records is not None:
    raw_train = raw_train[: CFG.debug_train_records]
if CFG.debug_dev_records is not None:
    raw_dev = raw_dev[: CFG.debug_dev_records]

print("Train:", train_stats)
print("Dev:  ", dev_stats)
print("Manifest SHA:", manifest_sha)

## 4. Детерминированный выбор synthetic data

Пути исправлены на текущую структуру `data/augmentation/...`.

- обычные augmentation-файлы читаются и сэмплируются по seed;
- `script_swap/part-*.jsonl` может быть очень большим, поэтому для него используется reservoir sampling без загрузки всего пула в RAM;
- если ratio > 0, а файла нет, notebook падает с понятной ошибкой вместо молчаливого изменения эксперимента.

In [ ]:
def deterministic_sample(records: Sequence[dict[str, Any]], n: int, seed: int) -> list[dict[str, Any]]:
    if n <= 0:
        return []
    rng = random.Random(seed)
    if n >= len(records):
        if n > len(records):
            warnings.warn(f"Запрошено {n}, доступно только {len(records)}; беру все")
        return list(records)
    indices = rng.sample(range(len(records)), n)
    return [records[i] for i in indices]


def reservoir_sample_jsonl(paths: Sequence[Path], n: int, seed: int) -> list[dict[str, Any]]:
    if n <= 0:
        return []
    rng = random.Random(seed)
    reservoir: list[dict[str, Any]] = []
    seen = 0
    for path in paths:
        with path.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                seen += 1
                if len(reservoir) < n:
                    reservoir.append(record)
                else:
                    j = rng.randrange(seen)
                    if j < n:
                        reservoir[j] = record
    if seen < n:
        warnings.warn(f"script_swap: запрошено {n}, доступно {seen}; беру все")
    return reservoir


def sample_source(path: Path, ratio: float, base_n: int, seed: int, name: str) -> list[dict[str, Any]]:
    target = int(round(base_n * ratio))
    if target == 0:
        return []
    if not path.exists():
        raise FileNotFoundError(
            f"Включён {name}_ratio={ratio}, но файл не найден: {path}. "
            "Сначала сгенерируйте augmentation или поставьте ratio=0.0"
        )
    records = read_jsonl(path)
    validate_records(records, name)
    return deterministic_sample(records, target, seed)

base_n = len(raw_train)
selected_by_source: dict[str, list[dict[str, Any]]] = {}
selected_by_source["real"] = list(raw_train)
selected_by_source["morphology"] = sample_source(
    AUG_DIR / "morphology.jsonl", CFG.morphology_ratio, base_n, CFG.seed + 11, "morphology"
)
selected_by_source["case"] = sample_source(
    AUG_DIR / "case.jsonl", CFG.case_ratio, base_n, CFG.seed + 23, "case"
)
selected_by_source["hard_negative"] = sample_source(
    AUG_DIR / "hard_negatives.jsonl", CFG.hard_negative_ratio, base_n, CFG.seed + 37, "hard_negative"
)

script_target = int(round(base_n * CFG.script_swap_ratio))
if script_target:
    script_paths = sorted((AUG_DIR / "script_swap").glob("part-*.jsonl"))
    if not script_paths:
        raise FileNotFoundError(
            f"Включён script_swap_ratio={CFG.script_swap_ratio}, но не найдены "
            f"{AUG_DIR / 'script_swap' / 'part-*.jsonl'}"
        )
    selected_by_source["script_swap"] = reservoir_sample_jsonl(
        script_paths, script_target, CFG.seed + 53
    )
    validate_records(selected_by_source["script_swap"], "script_swap")
else:
    selected_by_source["script_swap"] = []

selected_train = []
source_rows = []
for source, records in selected_by_source.items():
    selected_train.extend(records)
    source_rows.append({"source": source, "records": len(records)})

# Разные augmentation-скрипты должны выдавать уникальные hashes; проверяем после объединения.
validate_records(selected_train, "selected_train")

rng = random.Random(CFG.seed)
rng.shuffle(selected_train)

source_df = pd.DataFrame(source_rows)
display(source_df)
print("Итого selected train docs:", len(selected_train))

sampled_hashes = {
    source: [r["hash"] for r in records]
    for source, records in selected_by_source.items()
    if source != "real"
}
(ARTIFACT_DIR / "sampled_synthetic_hashes.json").write_text(
    json.dumps(sampled_hashes, indent=2, ensure_ascii=False), encoding="utf-8"
)

## 5. Splitter и audit exact boundaries

У публичного GLiNER2.5 checkpoint default splitter — whitespace. Для нашего exact-offset датасета это неудобно: пунктуация часто приклеена к слову и gold boundary оказывается внутри whitespace-token.

Поэтому по умолчанию используется punctuation-aware splitter:

- буквенно-цифровые последовательности и варианты узбекских апострофов остаются вместе;
- пунктуация становится отдельным word token;
- **тот же splitter обязательно передаётся при загрузке модели и при inference**, потому что splitter не сохраняется в checkpoint.

In [ ]:
class WhitespaceSplitter:
    _pattern = re.compile(r"\S+", re.UNICODE)

    def __call__(self, text: str, lower: bool = False):
        for m in self._pattern.finditer(text):
            token = m.group(0)
            yield (token.lower() if lower else token, m.start(), m.end())


class PunctuationAwareSplitter:
    # Сохраняем типичные апострофы внутри слова, остальные punctuation — отдельные tokens.
    _pattern = re.compile(r"[\wʻʼ’‘'`´]+|[^\w\s]", re.UNICODE)

    def __call__(self, text: str, lower: bool = False):
        for m in self._pattern.finditer(text):
            token = m.group(0)
            yield (token.lower() if lower else token, m.start(), m.end())


def make_splitter(name: str):
    if name == "punctuation_aware":
        return PunctuationAwareSplitter()
    if name == "whitespace":
        return WhitespaceSplitter()
    raise ValueError(f"Unknown word_splitter={name!r}")

WORD_SPLITTER = make_splitter(CFG.word_splitter)


def entity_token_interval(entity: dict[str, Any], tokens: Sequence[tuple[str, int, int]]):
    starts = {start: i for i, (_, start, _) in enumerate(tokens)}
    ends = {end: i for i, (_, _, end) in enumerate(tokens)}
    start, end = entity["start"], entity["end"]
    if start not in starts or end not in ends:
        return None
    i, j = starts[start], ends[end]
    return (i, j + 1) if i <= j else None


def audit_boundaries(records: Sequence[dict[str, Any]], splitter) -> dict[str, Any]:
    bad = []
    total = 0
    for record in records:
        tokens = list(splitter(record["text"], lower=True))
        for ent in record["entities"]:
            total += 1
            if entity_token_interval(ent, tokens) is None:
                bad.append({
                    "hash": record["hash"],
                    "label": ent["label"],
                    "start": ent["start"],
                    "end": ent["end"],
                    "surface": record["text"][ent["start"]:ent["end"]],
                })
    return {
        "entities": total,
        "bad": len(bad),
        "bad_rate": (len(bad) / total if total else 0.0),
        "examples": bad[:20],
    }

chosen_train_boundary_audit = audit_boundaries(selected_train, WORD_SPLITTER)
chosen_dev_boundary_audit = audit_boundaries(raw_dev, WORD_SPLITTER)
whitespace_train_audit = audit_boundaries(raw_train, WhitespaceSplitter())
whitespace_dev_audit = audit_boundaries(raw_dev, WhitespaceSplitter())

boundary_report = {
    "chosen_splitter": CFG.word_splitter,
    "selected_train": chosen_train_boundary_audit,
    "dev": chosen_dev_boundary_audit,
    "whitespace_reference_train": whitespace_train_audit,
    "whitespace_reference_dev": whitespace_dev_audit,
}

print(json.dumps(boundary_report, indent=2, ensure_ascii=False))

## 6. Безопасная конвертация offsets → GLiNER training examples

GLiNER training supervision задаётся примерно как:

```python
{
    "input": "...",
    "output": {
        "entities": {
            "organization": ["..."],
            "person": ["..."],
            "location": ["..."]
        }
    }
}
```

Внутри processor mention ищется в token sequence по тексту. Если одинаковый mention встречается дважды, а размечен только один, наивная конвертация создаст ложный target.

Поэтому функция ниже:

1. chunk'ит документ по word tokens и **не режет gold entity**;
2. проверяет все повторные occurrences для `(label, mention)`;
3. при неоднозначности рекурсивно делит chunk безопасной границей;
4. контролирует максимум gold spans одного label на chunk;
5. сохраняет negative chunks с пустыми списками для всех трёх типов;
6. при нерепрезентируемом character span по умолчанию исключает только соответствующий raw record из training conversion и явно пишет это в report.

In [ ]:
def _find_occurrences(seq: Sequence[str], sub: Sequence[str]) -> list[tuple[int, int]]:
    n = len(sub)
    if not n:
        return []
    return [(i, i + n) for i in range(len(seq) - n + 1) if list(seq[i:i+n]) == list(sub)]


def _safe_cut_positions(start: int, end: int, entities: Sequence[tuple]) -> list[int]:
    return [
        cut
        for cut in range(start + 1, end)
        if not any(ent_start < cut < ent_end for ent_start, ent_end, *_ in entities)
    ]


def _base_chunk_ranges(n_tokens: int, entities: Sequence[tuple], max_words: int) -> list[tuple[int, int]]:
    if n_tokens == 0:
        return []
    ranges = []
    start = 0
    while start < n_tokens:
        target = min(start + max_words, n_tokens)
        if target < n_tokens:
            crossing = [(s, e) for s, e, *_ in entities if s < target < e]
            if crossing:
                left = min(s for s, _ in crossing)
                target = left if left > start else max(e for _, e in crossing)
            if target <= start:
                target = min(start + max_words, n_tokens)
        ranges.append((start, target))
        start = target
    return ranges


def _segment_issue(
    start: int,
    end: int,
    tokens_lower: Sequence[str],
    entity_rows: Sequence[tuple],
    max_gold_per_label: int,
) -> str | None:
    inside = [x for x in entity_rows if start <= x[0] and x[1] <= end]

    counts = Counter(x[2] for x in inside)
    if any(v > max_gold_per_label for v in counts.values()):
        return "capacity"

    # GLiNER processor ищет ВСЕ token occurrences каждого mention.
    by_key: dict[tuple[str, tuple[str, ...]], set[tuple[int, int]]] = defaultdict(set)
    for ent_start, ent_end, model_label, *_ in inside:
        mention = tuple(tokens_lower[ent_start:ent_end])
        by_key[(model_label, mention)].add((ent_start, ent_end))

    local = list(tokens_lower[start:end])
    for (label, mention), gold_positions in by_key.items():
        all_positions = {
            (start + a, start + b)
            for a, b in _find_occurrences(local, mention)
        }
        if all_positions != gold_positions:
            return "ambiguous_mention"

    return None


def _recursive_make_safe(
    start: int,
    end: int,
    tokens_lower: Sequence[str],
    entity_rows: Sequence[tuple],
    max_gold_per_label: int,
) -> list[tuple[int, int, str | None]]:
    issue = _segment_issue(start, end, tokens_lower, entity_rows, max_gold_per_label)
    if issue is None:
        return [(start, end, None)]

    cuts = _safe_cut_positions(start, end, entity_rows)
    if not cuts or end - start <= 1:
        return [(start, end, issue)]

    mid = (start + end) // 2
    cut = min(cuts, key=lambda x: (abs(x - mid), x))
    if cut <= start or cut >= end:
        return [(start, end, issue)]

    return (
        _recursive_make_safe(start, cut, tokens_lower, entity_rows, max_gold_per_label)
        + _recursive_make_safe(cut, end, tokens_lower, entity_rows, max_gold_per_label)
    )


def record_to_gliner_segments(
    record: dict[str, Any],
    splitter,
    *,
    max_words: int,
    max_gold_per_label: int,
    use_descriptions: bool,
    unrepresentable_policy: str,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    text = record["text"]
    tokens = list(splitter(text, lower=False))
    tokens_lower = [t.lower() for t, _, _ in tokens]

    if not tokens:
        return [], {
            "dropped": True,
            "reason": "no_tokens",
            "gold_total": len(record["entities"]),
            "gold_kept": 0,
        }

    entity_rows = []
    bad = []
    for ent in record["entities"]:
        interval = entity_token_interval(ent, tokens)
        if interval is None:
            bad.append(ent)
            continue
        ent_start_tok, ent_end_tok = interval
        surface = text[ent["start"]:ent["end"]]
        entity_rows.append((
            ent_start_tok,
            ent_end_tok,
            DATA_TO_MODEL[ent["label"]],
            surface,
            ent["label"],
            ent["start"],
            ent["end"],
        ))

    if bad:
        if unrepresentable_policy == "raise":
            raise ValueError(
                f"{record['hash']}: {len(bad)} gold spans не совпадают с word boundaries"
            )
        if unrepresentable_policy == "drop_record":
            return [], {
                "dropped": True,
                "reason": "unrepresentable",
                "gold_total": len(record["entities"]),
                "gold_kept": 0,
                "bad": bad[:5],
            }
        raise ValueError(f"Unknown unrepresentable_policy={unrepresentable_policy!r}")

    basic_ranges = _base_chunk_ranges(len(tokens), entity_rows, max_words)
    safe_ranges = []
    for start, end in basic_ranges:
        safe_ranges.extend(
            _recursive_make_safe(
                start, end, tokens_lower, entity_rows, max_gold_per_label
            )
        )

    unresolved = [x for x in safe_ranges if x[2] is not None]
    if unresolved and unrepresentable_policy == "raise":
        raise ValueError(f"{record['hash']}: cannot safely segment: {unresolved[:3]}")

    resolved = [x for x in safe_ranges if x[2] is None]
    outputs = []
    gold_kept = 0

    for segment_index, (start, end, _) in enumerate(resolved):
        char_start = tokens[start][1]
        char_end = tokens[end - 1][2]
        segment_text = text[char_start:char_end]
        entities = {label: [] for label in MODEL_LABELS}

        for ent_start_tok, ent_end_tok, model_label, surface, _, char_s, char_e in entity_rows:
            if start <= ent_start_tok and ent_end_tok <= end:
                rel_s = char_s - char_start
                rel_e = char_e - char_start
                assert segment_text[rel_s:rel_e] == surface
                if surface not in entities[model_label]:
                    entities[model_label].append(surface)
                gold_kept += 1

        # Последняя защита: число occurrences mention в chunk должно совпадать с gold.
        segment_tokens_lower = tokens_lower[start:end]
        for model_label, mentions in entities.items():
            for mention in mentions:
                mention_tokens = [t for t, _, _ in splitter(mention, lower=True)]
                found = _find_occurrences(segment_tokens_lower, mention_tokens)
                expected = sum(
                    1
                    for ent_s, ent_e, label, *_ in entity_rows
                    if label == model_label
                    and tokens_lower[ent_s:ent_e] == mention_tokens
                    and start <= ent_s and ent_e <= end
                )
                if len(found) != expected:
                    raise AssertionError(
                        (record["hash"], model_label, mention, len(found), expected)
                    )

        output = {"entities": entities}
        if use_descriptions:
            output["entity_descriptions"] = ENTITY_DESCRIPTIONS.copy()

        outputs.append({
            "input": segment_text,
            "output": output,
            "_meta": {
                "source_hash": record["hash"],
                "segment_index": segment_index,
                "char_start": char_start,
                "char_end": char_end,
                "token_start": start,
                "token_end": end,
            },
        })

    dropped = not outputs
    reason = "ambiguous_unsplittable" if dropped and unresolved else None
    return outputs, {
        "dropped": dropped,
        "reason": reason,
        "gold_total": len(record["entities"]),
        "gold_kept": gold_kept,
        "segments": len(outputs),
        "unsafe_segments_dropped": len(unresolved),
        "unsafe_segment_examples": unresolved[:5],
    }


def convert_records(records: Sequence[dict[str, Any]], splitter, **kwargs):
    segments = []
    report = Counter()
    drop_examples = []
    gold_total = 0
    gold_kept = 0

    for record in records:
        record_segments, rep = record_to_gliner_segments(record, splitter, **kwargs)
        gold_total += rep["gold_total"]
        gold_kept += rep["gold_kept"]

        if rep["dropped"]:
            report["dropped_records"] += 1
            report[f"drop_{rep['reason']}"] += 1
            drop_examples.append({"hash": record["hash"], **rep})
        else:
            report["kept_records"] += 1
            report["segments"] += len(record_segments)
            segments.extend(record_segments)
            if rep.get("unsafe_segments_dropped", 0):
                report["partially_kept_records"] += 1
                report["unsafe_segments_dropped"] += rep["unsafe_segments_dropped"]

    return segments, {
        "records": len(records),
        **dict(report),
        "gold_total": gold_total,
        "gold_kept": gold_kept,
        "gold_retention": (gold_kept / gold_total if gold_total else 1.0),
        "drop_examples": drop_examples[:20],
    }

In [ ]:
train_examples, train_conversion_report = convert_records(
    selected_train,
    WORD_SPLITTER,
    max_words=CFG.train_chunk_words,
    max_gold_per_label=CFG.max_gold_per_label,
    use_descriptions=CFG.use_entity_descriptions,
    unrepresentable_policy=CFG.unrepresentable_policy,
)

dev_examples, dev_conversion_report = convert_records(
    raw_dev,
    WORD_SPLITTER,
    max_words=CFG.train_chunk_words,
    max_gold_per_label=CFG.max_gold_per_label,
    use_descriptions=CFG.use_entity_descriptions,
    unrepresentable_policy=CFG.unrepresentable_policy,
)

# GLiNER trainer не знает про наш _meta; в training input отдаём только input/output.
train_examples_for_model = [
    {"input": x["input"], "output": x["output"]}
    for x in train_examples
]
dev_examples_for_model = [
    {"input": x["input"], "output": x["output"]}
    for x in dev_examples
]

negative_train_segments = sum(
    1 for x in train_examples_for_model if not any(x["output"]["entities"].values())
)

prep_report = {
    "selected_sources": source_rows,
    "boundary_audit": boundary_report,
    "train_conversion": train_conversion_report,
    "dev_conversion": dev_conversion_report,
    "negative_train_segments": negative_train_segments,
    "train_examples": len(train_examples_for_model),
    "dev_examples": len(dev_examples_for_model),
}

(ARTIFACT_DIR / "data_prep_report.json").write_text(
    json.dumps(prep_report, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(prep_report, indent=2, ensure_ascii=False))

if dev_conversion_report["gold_retention"] < 1.0:
    raise RuntimeError(
        "DEV conversion потеряла gold spans. Для честной exact-span оценки это недопустимо."
    )

## 7. Загрузка GLiNER2.5 Multi и preflight

Используем `AutoExtractor`, потому что GLiNER2.5 — **boundary checkpoint**; legacy span-only loader для него не подходит.

Также отключаем stochastic schema-label augmentation самого GLiNER processor: у нас фиксированная схема из трёх классов, а внешние synthetic-эксперименты должны контролироваться только через `ExperimentConfig`.

In [ ]:
try:
    import torch
    import gliner2
    from gliner2 import AutoExtractor
    from gliner2.processor import SamplingConfig
    from gliner2.training.trainer import ExtractorTrainer, TrainingConfig
except ImportError as e:
    raise ImportError(
        "Не установлены зависимости GLiNER2. Запустите install-cell выше и перезапустите kernel."
    ) from e

print("torch:", torch.__version__)
print("gliner2:", getattr(gliner2, "__version__", "unknown"))


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    device = torch.device(name)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CFG.device='cuda', но CUDA недоступна")
    if device.type == "mps":
        if not getattr(torch.backends, "mps", None) or not torch.backends.mps.is_available():
            raise RuntimeError("CFG.device='mps', но MPS недоступен")
    return device


def resolve_precision(name: str, device: torch.device) -> tuple[bool, bool, str]:
    if name == "auto":
        # Boundary model предпочитает BF16 на совместимой CUDA. На MPS/CPU — FP32.
        if device.type == "cuda" and torch.cuda.is_bf16_supported():
            return False, True, "bf16"
        return False, False, "fp32"
    if name == "fp32":
        return False, False, "fp32"
    if name == "bf16":
        if device.type != "cuda" or not torch.cuda.is_bf16_supported():
            raise RuntimeError("BF16 в этом notebook разрешён только на совместимой CUDA")
        return False, True, "bf16"
    if name == "fp16":
        if device.type != "cuda":
            raise RuntimeError("FP16 training в этом notebook разрешён только на CUDA")
        return True, False, "fp16"
    raise ValueError("precision: auto | fp32 | fp16 | bf16")

DEVICE = resolve_device(CFG.device)
FP16, BF16, PRECISION_NAME = resolve_precision(CFG.precision, DEVICE)

model = AutoExtractor.from_pretrained(
    CFG.model_name,
    map_location="cpu",
    word_splitter=WORD_SPLITTER,
)

architecture = getattr(model, "architecture", getattr(getattr(model, "config", None), "architecture", None))
print("Architecture:", architecture)
print("Device:", DEVICE, "| precision:", PRECISION_NAME)

if CFG.require_boundary_architecture and architecture != "boundary":
    raise RuntimeError(
        f"Ожидался GLiNER2.5 boundary checkpoint, получено architecture={architecture!r}"
    )

# Фиксированная NER schema: отключаем рандомное добавление synthetic entity labels.
model.processor.sampling_config = SamplingConfig(
    remove_entities_prob=0.0,
    shuffle_entities=False,
    remove_entity_prob=0.0,
    synthetic_entity_label_prob=0.0,
)

# Проверяем несколько examples через реальный processor до запуска долгого training.
preflight_n = min(CFG.preflight_examples, len(train_examples_for_model))
preflight_rng = random.Random(CFG.seed)
preflight_indices = preflight_rng.sample(range(len(train_examples_for_model)), preflight_n)
preflight_lengths = []

for idx in preflight_indices:
    ex = train_examples_for_model[idx]
    transformed = model.processor.transform_and_format(ex["input"], ex["output"])
    preflight_lengths.append(len(transformed.input_ids))

encoder = getattr(model, "encoder", None)
encoder_config = getattr(encoder, "config", None)
max_positions = getattr(encoder_config, "max_position_embeddings", None)

print(
    f"Preflight encoder lengths: n={len(preflight_lengths)}, "
    f"max={max(preflight_lengths)}, p95={pd.Series(preflight_lengths).quantile(0.95):.0f}, "
    f"encoder max_position_embeddings={max_positions}"
)

if isinstance(max_positions, int) and max_positions > 0 and max(preflight_lengths) > max_positions:
    raise RuntimeError(
        "Некоторые training examples после добавления schema превышают encoder context. "
        "Уменьшите CFG.train_chunk_words."
    )

## 8. Trainer: early stopping, best/final, MPS workaround

В текущем GLiNER2 trainer CUDA выбирается автоматически, иначе fallback идёт на CPU. Для Apple Silicon ниже есть небольшой single-device subclass, который явно включает `mps`.

Другие исправления относительно старого notebook:

- eval **1 раз на эпоху**, а не 3;
- early stopping;
- best checkpoint по `eval_loss`;
- `save_total_limit=1` для промежуточных checkpoint'ов;
- нет второго ручного сохранения тех же весов в `training_state.pt`;
- `best/` и `final/` создаёт native trainer;
- mixed precision задаётся **явно**, поэтому GLiNER2 не включит неподдерживаемый BF16/FP16 автоматически.

In [ ]:
class SingleDeviceExtractorTrainer(ExtractorTrainer):
    """ExtractorTrainer с явным single-device выбором CUDA/MPS/CPU."""

    forced_device: torch.device = DEVICE

    def _setup_device(self):
        self.device = self.forced_device
        self.is_distributed = False

        if self.device.type != "cuda" and (self.config.fp16 or self.config.bf16):
            warnings.warn(f"Mixed precision отключена на {self.device}")
            self.config.fp16 = False
            self.config.bf16 = False

        if self.device.type == "mps":
            # Надёжные настройки для macOS DataLoader / optimizer.
            self.config.num_workers = 0
            self.config.pin_memory = False
            self.config.fused_optimizer = False

        self.model.to(self.device)
        if not self.config.fp16 and not self.config.bf16:
            self.model.float()

        print(f"Trainer device: {self.device}")

training_config = TrainingConfig(
    output_dir=str(TRAINING_DIR),
    experiment_name=EXPERIMENT_NAME,
    num_epochs=CFG.num_epochs,
    batch_size=CFG.train_batch_size,
    eval_batch_size=CFG.eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    encoder_lr=CFG.encoder_lr,
    task_lr=CFG.task_lr,
    weight_decay=CFG.weight_decay,
    max_grad_norm=CFG.max_grad_norm,
    scheduler_type=CFG.scheduler_type,
    warmup_ratio=CFG.warmup_ratio,

    # Всегда передаём explicit bools: это важно для boundary architecture.
    fp16=FP16,
    bf16=BF16,

    eval_strategy=CFG.eval_strategy,
    save_total_limit=CFG.save_total_limit,
    save_best=True,
    metric_for_best="eval_loss",
    greater_is_better=False,
    early_stopping=CFG.early_stopping,
    early_stopping_patience=CFG.early_stopping_patience,

    logging_steps=10,
    num_workers=0 if DEVICE.type == "mps" else 2,
    pin_memory=(DEVICE.type == "cuda"),
    seed=CFG.seed,
    deterministic=False,

    validate_data=True,
    max_len=CFG.train_chunk_words,
    strict_training=True,
    on_capacity_exceeded="raise",
    group_by_length=True,
    gradient_checkpointing=CFG.gradient_checkpointing,
    fused_optimizer=False,

    use_lora=CFG.use_lora,
    lora_r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    lora_target_modules=list(CFG.lora_target_modules),
    save_adapter_only=CFG.save_adapter_only,
)

print(training_config)

## 9. Fine-tuning

> Эта ячейка запускает долгий training.

Для быстрого smoke-test перед полным запуском можно временно поставить в config:

```python
CFG.debug_train_records = 128
CFG.debug_dev_records = 64
CFG.num_epochs = 1
```

и выполнить notebook сверху заново.

In [ ]:
trainer = SingleDeviceExtractorTrainer(model=model, config=training_config)

started_at = time.time()
training_result = trainer.train(
    train_data=train_examples_for_model,
    eval_data=dev_examples_for_model,
)
elapsed_sec = time.time() - started_at


def to_jsonable(value: Any):
    if dataclasses.is_dataclass(value):
        return {k: to_jsonable(v) for k, v in dataclasses.asdict(value).items()}
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    if isinstance(value, Path):
        return str(value)
    return value

training_summary = {
    "elapsed_sec": elapsed_sec,
    "result": to_jsonable(training_result),
    "train_metrics_history": to_jsonable(trainer.train_metrics_history),
    "eval_metrics_history": to_jsonable(trainer.eval_metrics_history),
}

(ARTIFACT_DIR / "training_history.json").write_text(
    json.dumps(training_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

BEST_DIR = TRAINING_DIR / "best"
FINAL_DIR = TRAINING_DIR / "final"

if not BEST_DIR.exists():
    raise FileNotFoundError(f"Best checkpoint не найден: {BEST_DIR}")

print(f"Training finished in {elapsed_sec / 60:.1f} min")
print("Best checkpoint:", BEST_DIR)
print("Final checkpoint:", FINAL_DIR if FINAL_DIR.exists() else "not found")

## 10. Reload best checkpoint и сбор low-threshold dev candidates

Threshold tuning не требует повторного forward-pass для каждой комбинации порогов:

1. один раз извлекаем dev entities с низким `proposal_threshold`;
2. сохраняем `(label, start, end, confidence)`;
3. дальше перебираем thresholds только на CPU по сохранённым кандидатам.

Для длинных документов используется native `extract_entities_long`, который возвращает глобальные character offsets.

In [ ]:
# Освобождаем training model перед inference/reload.
try:
    del trainer
    del model
except NameError:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()
if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    torch.mps.empty_cache()

best_model = AutoExtractor.from_pretrained(
    str(BEST_DIR),
    map_location="cpu",
    word_splitter=WORD_SPLITTER,  # splitter runtime-only, поэтому передаём снова
)
best_model.to(DEVICE)
best_model.eval()
if PRECISION_NAME == "fp32":
    best_model.float()

sig = inspect.signature(best_model.extract_entities_long)
print("extract_entities_long signature:", sig)
if "threshold" not in sig.parameters and not any(
    p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()
):
    raise RuntimeError("Текущая версия GLiNER2 не принимает threshold в extract_entities_long")

INFERENCE_SCHEMA = ENTITY_DESCRIPTIONS if CFG.use_entity_descriptions else list(MODEL_LABELS)


def parse_entity_candidates(result: dict[str, Any], text: str) -> list[dict[str, Any]]:
    out = []
    entities_dict = result.get("entities", {})
    for model_label, items in entities_dict.items():
        if model_label not in MODEL_TO_DATA:
            continue
        if not isinstance(items, list):
            continue
        for item in items:
            if not isinstance(item, dict):
                # При include_spans/confidence=True ожидаем dict; иначе это несовместимый API.
                raise TypeError(f"Unexpected entity item: {item!r}")
            start = int(item["start"])
            end = int(item["end"])
            confidence = float(item["confidence"])
            if not 0 <= start < end <= len(text):
                raise ValueError(f"Bad predicted span {start}:{end} for len={len(text)}")
            returned_text = item.get("text")
            source_text = text[start:end]
            if returned_text is not None and returned_text != source_text:
                warnings.warn(
                    f"GLiNER returned text/span mismatch: {returned_text!r} != {source_text!r}; "
                    "для exact-span используем offsets + source slice"
                )
            out.append({
                "label": MODEL_TO_DATA[model_label],
                "start": start,
                "end": end,
                "confidence": confidence,
                "text": source_text,
            })

    # Дубликаты из overlap chunks: оставляем max confidence.
    best = {}
    for item in out:
        key = (item["label"], item["start"], item["end"])
        if key not in best or item["confidence"] > best[key]["confidence"]:
            best[key] = item
    return list(best.values())


def collect_dev_candidates(model, records: Sequence[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:
    all_candidates = {}
    for i, record in enumerate(records, 1):
        result = model.extract_entities_long(
            record["text"],
            INFERENCE_SCHEMA,
            chunk_size=CFG.inference_chunk_words,
            chunk_overlap=CFG.inference_chunk_overlap,
            include_spans=True,
            include_confidence=True,
            threshold=CFG.proposal_threshold,
            overlap_policy="allow",
        )
        all_candidates[record["hash"]] = parse_entity_candidates(result, record["text"])
        if i % 100 == 0 or i == len(records):
            print(f"Inference: {i}/{len(records)}")
    return all_candidates

DEV_CANDIDATES = collect_dev_candidates(best_model, raw_dev)

with (ARTIFACT_DIR / "dev_candidates.jsonl").open("w", encoding="utf-8") as f:
    for record in raw_dev:
        f.write(json.dumps({
            "hash": record["hash"],
            "candidates": DEV_CANDIDATES[record["hash"]],
        }, ensure_ascii=False) + "\n")

num_candidates = sum(map(len, DEV_CANDIDATES.values()))
print("Saved dev candidates:", num_candidates)

## 11. Exact-span scorer + deterministic overlap resolution

Threshold применяется **после** entity decoding на native GLiNER confidence, а не к отдельным BIO tokens.

После threshold filtering пересечения разрешаются greedy-алгоритмом:

1. выше confidence;
2. при равенстве — длиннее span;
3. затем меньший `start`;
4. затем label.

Gold в нашем датасете непересекающийся, поэтому финальные predictions тоже делаем flat/non-overlapping.

In [ ]:
def spans_overlap(a: dict[str, Any], b: dict[str, Any]) -> bool:
    return a["start"] < b["end"] and b["start"] < a["end"]


def resolve_flat_greedy(candidates: Sequence[dict[str, Any]]) -> list[dict[str, Any]]:
    ordered = sorted(
        candidates,
        key=lambda x: (
            -x["confidence"],
            -(x["end"] - x["start"]),
            x["start"],
            x["end"],
            x["label"],
        ),
    )
    kept = []
    for cand in ordered:
        if not any(spans_overlap(cand, prev) for prev in kept):
            kept.append(cand)
    return sorted(kept, key=lambda x: (x["start"], x["end"], x["label"]))


def apply_thresholds(
    records: Sequence[dict[str, Any]],
    candidates_by_hash: dict[str, list[dict[str, Any]]],
    thresholds: dict[str, float],
) -> list[dict[str, Any]]:
    predictions = []
    for record in records:
        filtered = [
            x for x in candidates_by_hash[record["hash"]]
            if x["confidence"] >= thresholds[x["label"]]
        ]
        final = resolve_flat_greedy(filtered)
        predictions.append({
            "hash": record["hash"],
            "entities": [
                {"label": x["label"], "start": x["start"], "end": x["end"]}
                for x in final
            ],
        })
    return predictions


def exact_span_metrics(
    gold_records: Sequence[dict[str, Any]],
    pred_records: Sequence[dict[str, Any]],
) -> dict[str, Any]:
    gold_by_hash = {r["hash"]: r for r in gold_records}
    pred_by_hash = {r["hash"]: r for r in pred_records}
    if set(gold_by_hash) != set(pred_by_hash):
        missing = set(gold_by_hash) - set(pred_by_hash)
        extra = set(pred_by_hash) - set(gold_by_hash)
        raise ValueError(f"Prediction hashes mismatch: missing={len(missing)}, extra={len(extra)}")

    per_label = {}
    micro_tp = micro_fp = micro_fn = 0

    for label in LABELS:
        tp = fp = fn = 0
        for h, gold in gold_by_hash.items():
            pred = pred_by_hash[h]
            gold_set = {
                (e["start"], e["end"])
                for e in gold["entities"] if e["label"] == label
            }
            pred_set = {
                (e["start"], e["end"])
                for e in pred["entities"] if e["label"] == label
            }
            tp += len(gold_set & pred_set)
            fp += len(pred_set - gold_set)
            fn += len(gold_set - pred_set)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        micro_tp += tp
        micro_fp += fp
        micro_fn += fn

    micro_p = micro_tp / (micro_tp + micro_fp) if micro_tp + micro_fp else 0.0
    micro_r = micro_tp / (micro_tp + micro_fn) if micro_tp + micro_fn else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    macro_f1 = sum(per_label[x]["f1"] for x in LABELS) / len(LABELS)

    return {
        "micro": {
            "tp": micro_tp,
            "fp": micro_fp,
            "fn": micro_fn,
            "precision": micro_p,
            "recall": micro_r,
            "f1": micro_f1,
        },
        "macro_f1": macro_f1,
        "per_label": per_label,
    }


def threshold_grid(cfg: ExperimentConfig) -> list[float]:
    n = int(round((cfg.threshold_end - cfg.threshold_start) / cfg.threshold_step))
    values = [round(cfg.threshold_start + i * cfg.threshold_step, 10) for i in range(n + 1)]
    return [x for x in values if x <= cfg.threshold_end + 1e-12]

DEFAULT_THRESHOLDS = {label: CFG.default_threshold for label in LABELS}
default_predictions = apply_thresholds(raw_dev, DEV_CANDIDATES, DEFAULT_THRESHOLDS)
default_metrics = exact_span_metrics(raw_dev, default_predictions)

print("Default thresholds:", DEFAULT_THRESHOLDS)
print(json.dumps(default_metrics, indent=2, ensure_ascii=False))

## 12. Threshold search

Сначала находим лучший **global threshold**. Затем:

- `coordinate` (default): по очереди оптимизируем `ORG`, `NAME`, `GEO` несколько rounds;
- `exhaustive`: полный Cartesian product `grid³` — медленнее, но гарантирует лучший вариант на данной сетке.

Tie-break при одинаковом F1 предпочитает thresholds ближе к 0.5, чтобы не выбирать экстремальный порог без реального выигрыша.

> **Методологическая оговорка:** tuned dev F1 — это уже метрика после подбора гиперпараметра на dev. Для окончательной честной оценки нужен скрытый test / отдельный holdout.

In [ ]:
GRID = threshold_grid(CFG)
search_rows = []


def evaluate_thresholds(thresholds: dict[str, float], stage: str) -> tuple[float, dict[str, Any]]:
    preds = apply_thresholds(raw_dev, DEV_CANDIDATES, thresholds)
    metrics = exact_span_metrics(raw_dev, preds)
    row = {
        "stage": stage,
        **{f"thr_{label}": thresholds[label] for label in LABELS},
        "micro_f1": metrics["micro"]["f1"],
        "precision": metrics["micro"]["precision"],
        "recall": metrics["micro"]["recall"],
        **{f"f1_{label}": metrics["per_label"][label]["f1"] for label in LABELS},
    }
    search_rows.append(row)
    return metrics["micro"]["f1"], metrics


def better_candidate(score: float, thresholds: dict[str, float], best_score: float, best_thresholds: dict[str, float]) -> bool:
    if score > best_score + 1e-12:
        return True
    if abs(score - best_score) <= 1e-12:
        current_distance = sum(abs(thresholds[x] - 0.5) for x in LABELS)
        best_distance = sum(abs(best_thresholds[x] - 0.5) for x in LABELS)
        return current_distance < best_distance - 1e-12
    return False

# 1) Global threshold initialization.
best_score = -1.0
best_thresholds = DEFAULT_THRESHOLDS.copy()
for threshold in GRID:
    thresholds = {label: threshold for label in LABELS}
    score, _ = evaluate_thresholds(thresholds, "global")
    if better_candidate(score, thresholds, best_score, best_thresholds):
        best_score = score
        best_thresholds = thresholds.copy()

print("Best global:", best_thresholds, "F1=", best_score)

# 2) Per-label search.
if CFG.threshold_search_mode == "coordinate":
    for round_idx in range(CFG.threshold_coordinate_rounds):
        changed = False
        for label in LABELS:
            local_best_score = best_score
            local_best = best_thresholds.copy()
            for threshold in GRID:
                trial = best_thresholds.copy()
                trial[label] = threshold
                score, _ = evaluate_thresholds(trial, f"coord_r{round_idx+1}_{label}")
                if better_candidate(score, trial, local_best_score, local_best):
                    local_best_score = score
                    local_best = trial.copy()
            if local_best != best_thresholds:
                changed = True
            best_score = local_best_score
            best_thresholds = local_best
        if not changed:
            break
else:
    for values in itertools.product(GRID, repeat=len(LABELS)):
        trial = dict(zip(LABELS, values))
        score, _ = evaluate_thresholds(trial, "exhaustive")
        if better_candidate(score, trial, best_score, best_thresholds):
            best_score = score
            best_thresholds = trial.copy()

TUNED_THRESHOLDS = best_thresholds
TUNED_PREDICTIONS = apply_thresholds(raw_dev, DEV_CANDIDATES, TUNED_THRESHOLDS)
TUNED_METRICS = exact_span_metrics(raw_dev, TUNED_PREDICTIONS)

search_df = pd.DataFrame(search_rows).drop_duplicates(
    subset=[f"thr_{x}" for x in LABELS], keep="first"
).sort_values("micro_f1", ascending=False)

search_df.to_csv(ARTIFACT_DIR / "threshold_search.csv", index=False)
(ARTIFACT_DIR / "best_thresholds.json").write_text(
    json.dumps({
        "thresholds": TUNED_THRESHOLDS,
        "metrics": TUNED_METRICS,
        "proposal_threshold": CFG.proposal_threshold,
        "mode": CFG.threshold_search_mode,
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(search_df.head(20))
print("Tuned thresholds:", TUNED_THRESHOLDS)
print(json.dumps(TUNED_METRICS, indent=2, ensure_ascii=False))

## 13. Сохранение predictions + official scorer

Сохраняем два варианта:

- `dev_predictions_default.jsonl` — threshold 0.5;
- `dev_predictions_tuned.jsonl` — лучшие per-label thresholds.

Затем, если `scripts/evaluate.py` существует, прогоняем официальный scorer и проверяем, что его micro-F1 совпадает с локальной exact-span реализацией.

In [ ]:
def write_predictions(path: Path, predictions: Sequence[dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in predictions:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

DEFAULT_PRED_PATH = ARTIFACT_DIR / "dev_predictions_default.jsonl"
TUNED_PRED_PATH = ARTIFACT_DIR / "dev_predictions_tuned.jsonl"
DEFAULT_METRICS_PATH = ARTIFACT_DIR / "dev_metrics_default.json"
TUNED_METRICS_PATH = ARTIFACT_DIR / "dev_metrics_tuned.json"

write_predictions(DEFAULT_PRED_PATH, default_predictions)
write_predictions(TUNED_PRED_PATH, TUNED_PREDICTIONS)
DEFAULT_METRICS_PATH.write_text(json.dumps(default_metrics, indent=2), encoding="utf-8")
TUNED_METRICS_PATH.write_text(json.dumps(TUNED_METRICS, indent=2), encoding="utf-8")


def run_official_scorer(pred_path: Path, suffix: str) -> dict[str, Any] | None:
    scorer = PROJECT_DIR / "scripts" / "evaluate.py"
    if not scorer.exists():
        warnings.warn(f"Official scorer не найден: {scorer}")
        return None

    out_path = ARTIFACT_DIR / f"official_metrics_{suffix}.json"
    command = [
        sys.executable,
        str(scorer),
        "--gold", str(DEV_PATH),
        "--predictions", str(pred_path),
        "--output", str(out_path),
    ]
    result = subprocess.run(command, cwd=PROJECT_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Official scorer failed with code {result.returncode}")
    return json.loads(out_path.read_text(encoding="utf-8"))


def find_micro_f1(obj: Any) -> float | None:
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in {"f1_micro", "micro_f1"} and isinstance(value, (int, float)):
                return float(value)
        if "micro" in obj and isinstance(obj["micro"], dict):
            for key in ("f1", "f1_micro", "micro_f1"):
                if key in obj["micro"] and isinstance(obj["micro"][key], (int, float)):
                    return float(obj["micro"][key])
        for value in obj.values():
            found = find_micro_f1(value)
            if found is not None:
                return found
    return None

if CFG.run_official_scorer:
    official_default = run_official_scorer(DEFAULT_PRED_PATH, "default")
    official_tuned = run_official_scorer(TUNED_PRED_PATH, "tuned")

    for name, official, local in [
        ("default", official_default, default_metrics),
        ("tuned", official_tuned, TUNED_METRICS),
    ]:
        official_f1 = find_micro_f1(official)
        if official_f1 is not None:
            local_f1 = local["micro"]["f1"]
            if not math.isclose(official_f1, local_f1, rel_tol=0, abs_tol=1e-9):
                raise AssertionError(
                    f"{name}: local micro-F1={local_f1} != official={official_f1}"
                )
else:
    official_default = official_tuned = None

print("Saved:", DEFAULT_PRED_PATH)
print("Saved:", TUNED_PRED_PATH)

## 14. Error analysis tuned predictions

Категории:

- `wrong_label` — exact same `(start, end)`, но другой label;
- `boundary` — тот же label и пересекающийся, но не exact span;
- `missed` — gold не сопоставился;
- `spurious` — prediction не сопоставился.

Greedy matching используется только для аналитики; official score остаётся строгим exact matching.

In [ ]:
def classify_errors(
    gold_records: Sequence[dict[str, Any]],
    pred_records: Sequence[dict[str, Any]],
    context_chars: int = 80,
) -> pd.DataFrame:
    pred_by_hash = {r["hash"]: r for r in pred_records}
    rows = []

    for gold_record in gold_records:
        h = gold_record["hash"]
        text = gold_record["text"]
        gold = [dict(e) for e in gold_record["entities"]]
        pred = [dict(e) for e in pred_by_hash[h]["entities"]]

        matched_g = set()
        matched_p = set()

        # exact correct
        for gi, g in enumerate(gold):
            for pi, p in enumerate(pred):
                if pi in matched_p:
                    continue
                if (g["label"], g["start"], g["end"]) == (p["label"], p["start"], p["end"]):
                    matched_g.add(gi)
                    matched_p.add(pi)
                    break

        # same exact span, wrong label
        for gi, g in enumerate(gold):
            if gi in matched_g:
                continue
            for pi, p in enumerate(pred):
                if pi in matched_p:
                    continue
                if (g["start"], g["end"]) == (p["start"], p["end"]) and g["label"] != p["label"]:
                    matched_g.add(gi)
                    matched_p.add(pi)
                    rows.append((h, "wrong_label", g, p, text))
                    break

        # overlapping same-label = boundary error; choose max intersection.
        for gi, g in enumerate(gold):
            if gi in matched_g:
                continue
            options = []
            for pi, p in enumerate(pred):
                if pi in matched_p or p["label"] != g["label"]:
                    continue
                inter = max(0, min(g["end"], p["end"]) - max(g["start"], p["start"]))
                if inter > 0:
                    options.append((inter, pi, p))
            if options:
                _, pi, p = max(options, key=lambda x: (x[0], -abs((x[2]["end"]-x[2]["start"]) - (g["end"]-g["start"]))))
                matched_g.add(gi)
                matched_p.add(pi)
                rows.append((h, "boundary", g, p, text))

        for gi, g in enumerate(gold):
            if gi not in matched_g:
                rows.append((h, "missed", g, None, text))
        for pi, p in enumerate(pred):
            if pi not in matched_p:
                rows.append((h, "spurious", None, p, text))

    formatted = []
    for h, kind, g, p, text in rows:
        starts = [x["start"] for x in (g, p) if x is not None]
        ends = [x["end"] for x in (g, p) if x is not None]
        left = max(0, min(starts) - context_chars)
        right = min(len(text), max(ends) + context_chars)
        formatted.append({
            "hash": h,
            "error_type": kind,
            "gold_label": None if g is None else g["label"],
            "gold_start": None if g is None else g["start"],
            "gold_end": None if g is None else g["end"],
            "gold_text": None if g is None else text[g["start"]:g["end"]],
            "pred_label": None if p is None else p["label"],
            "pred_start": None if p is None else p["start"],
            "pred_end": None if p is None else p["end"],
            "pred_text": None if p is None else text[p["start"]:p["end"]],
            "context": text[left:right],
        })

    return pd.DataFrame(formatted)

errors_df = classify_errors(raw_dev, TUNED_PREDICTIONS)
errors_df.to_csv(ARTIFACT_DIR / "error_analysis.csv", index=False)

display(errors_df["error_type"].value_counts().rename_axis("error_type").to_frame("count"))
display(errors_df.head(30))

## 15. Итог эксперимента

Финальный summary фиксирует настройки, default/tuned F1 и пути к best checkpoint. Это позволяет потом собрать таблицу сетки экспериментов без ручного копирования результатов из output ячеек.

In [ ]:
summary = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": CFG.model_name,
    "architecture": architecture,
    "device": str(DEVICE),
    "precision": PRECISION_NAME,
    "best_checkpoint": str(BEST_DIR),
    "final_checkpoint": str(FINAL_DIR),
    "selected_train_docs": len(selected_train),
    "train_examples": len(train_examples_for_model),
    "dev_examples": len(dev_examples_for_model),
    "default_thresholds": DEFAULT_THRESHOLDS,
    "default_metrics": default_metrics,
    "tuned_thresholds": TUNED_THRESHOLDS,
    "tuned_metrics": TUNED_METRICS,
    "official_default": official_default,
    "official_tuned": official_tuned,
}

(ARTIFACT_DIR / "summary.json").write_text(
    json.dumps(to_jsonable(summary), indent=2, ensure_ascii=False), encoding="utf-8"
)

rows = []
for name, thresholds, metrics in [
    ("default", DEFAULT_THRESHOLDS, default_metrics),
    ("tuned", TUNED_THRESHOLDS, TUNED_METRICS),
]:
    row = {
        "variant": name,
        **{f"thr_{label}": thresholds[label] for label in LABELS},
        "precision_micro": metrics["micro"]["precision"],
        "recall_micro": metrics["micro"]["recall"],
        "f1_micro": metrics["micro"]["f1"],
        "f1_macro": metrics["macro_f1"],
        **{f"f1_{label}": metrics["per_label"][label]["f1"] for label in LABELS},
    }
    rows.append(row)

display(pd.DataFrame(rows))
print("Artifacts:", ARTIFACT_DIR)

## Рекомендуемая первая сетка экспериментов

Не меняйте сразу несколько осей, если хотите понимать источник прироста.

| Exp | Model | Morphology | Case | Hard neg | Threshold tuning |
|---|---|---:|---:|---:|---|
| G0 | GLiNER2.5 Multi | 0.00 | 0.00 | 0.00 | yes |
| G1 | GLiNER2.5 Multi | 0.25 | 0.00 | 0.00 | yes |
| G2 | GLiNER2.5 Multi | 0.50 | 0.00 | 0.00 | yes |
| G3 | GLiNER2.5 Multi | 0.25 | 0.25 | 0.00 | yes |
| G4 | GLiNER2.5 Multi | 0.25 | 0.00 | 0.01 | yes |

После этого имеет смысл отдельно сравнить full fine-tuning vs LoRA и только затем расширять synthetic/grid.

### Что считать результатом

Главная метрика для сравнения с текущим baseline — **official exact-span micro-F1** на том же `dev.jsonl`.

`tuned` результат используйте как рабочую конфигурацию inference; `default` сохраняется, чтобы видеть, сколько именно дал threshold tuning.